In [237]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [238]:
! kaggle competitions download -c house-prices-advanced-regression-techniques

house-prices-advanced-regression-techniques.zip: Skipping, found more recently modified local copy (use --force to force download)


In [239]:
import zipfile

zip_path = "house-prices-advanced-regression-techniques.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("house_prices")

print("Dataset Extracted Successfully!")

Dataset Extracted Successfully!


In [240]:
import os

print(os.listdir("house_prices"))

['data_description.txt', 'sample_submission.csv', 'test.csv', 'train.csv']


## First Dataset


In [241]:


df = pd.read_csv("house_prices/train.csv")


print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [242]:
df.dtypes

Id                 int64
MSSubClass         int64
MSZoning             str
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType             str
SaleCondition        str
SalePrice          int64
Length: 81, dtype: object

## Second Dataset

In [243]:
data=sns.load_dataset("titanic")
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## PAART 1 DATA QUALITY

## Check Missing Values 

In [244]:
print("--- Missing Values in First Dataset---")
print(df.isnull().sum().sort_values(ascending=False))
# print(("Missing Values in Second dataset---"))
# print(data.isnull().sum().sort_values(ascending=False))

--- Missing Values in First Dataset---
PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
                 ... 
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
SalePrice           0
Length: 81, dtype: int64


## Check Duplicate Records

In [245]:
print(("Duplicate Records in First dataset---"))
print(df.duplicated().sort_values(ascending=False))
print(("Duplicate Records in Second dataset---"))
print(data.duplicated().sort_values(ascending=False))

Duplicate Records in First dataset---
0       False
981     False
979     False
978     False
977     False
        ...  
484     False
483     False
482     False
481     False
1459    False
Length: 1460, dtype: bool
Duplicate Records in Second dataset---
466     True
733     True
133     True
358     True
355     True
       ...  
305    False
306    False
307    False
308    False
890    False
Length: 891, dtype: bool


## Drop duplicated Data


In [246]:
print("--- Drop duplicated data from First Dataset---")
print(df.drop_duplicates(inplace=True))
print("--- Drop duplicated data from Second Dataset---")
print(data.drop_duplicates(inplace=True))

--- Drop duplicated data from First Dataset---
None
--- Drop duplicated data from Second Dataset---
None


## Invalid Values

In [247]:
print("--- Check Invalid values in First Dataset---")
# invalid_sal=(df['SalePrice']<=0).sum()
# if invalid_sal<0:
#     print("Invalid Salary")
# else:
#     print("Not Invalid saleprice")
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
neg_columns_found = 0

print("\n. Checking for Negative Values across all numerical columns:")
for col in numerical_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"  Column '{col}' have {neg_count} negative values!")
        neg_columns_found += 1

if neg_columns_found == 0:
    print("   Data safe No invalid Values in dataset  ")


print("--- Invalid values in Second Dataset")

numeric_co=data.select_dtypes(include=['int64','float64']).columns
for col in numeric_co:
    neg_cont=(data[col]<0).sum()
    if neg_cont>0:
        print(f"Invalid data {neg_cont} in {col} column")
else:
    print("Data Safe")




# invalid_age_count = (data['age'] < 0).sum()
# if invalid_age_count<0:
#     print("Invalid Values")
# else:
#     print("No Invalid Values")

--- Check Invalid values in First Dataset---

. Checking for Negative Values across all numerical columns:
   Data safe No invalid Values in dataset  
--- Invalid values in Second Dataset
Data Safe


##  Inconsistent categories

In [248]:
print("--- Categorical Consistency Report of First Dataset---")
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    
    unique_vals = df[col].unique()
    
    print(f"\nColumn '{col}' has {len(unique_vals)} unique categories:")

    print(unique_vals[:5])

print("--- Categorical Consistency Report of Second Dataset---") 
cat_col=(data.select_dtypes(include=['object'])).columns

for col in cat_col:
    unique_val=data[col].unique()
    print(f"\nColumn '{col}' has {len(unique_vals)} unique categories:")

    print(unique_val[:5])



--- Categorical Consistency Report of First Dataset---

Column 'MSZoning' has 5 unique categories:
<ArrowStringArray>
['RL', 'RM', 'C (all)', 'FV', 'RH']
Length: 5, dtype: str

Column 'Street' has 2 unique categories:
<ArrowStringArray>
['Pave', 'Grvl']
Length: 2, dtype: str

Column 'Alley' has 3 unique categories:
<ArrowStringArray>
[nan, 'Grvl', 'Pave']
Length: 3, dtype: str

Column 'LotShape' has 4 unique categories:
<ArrowStringArray>
['Reg', 'IR1', 'IR2', 'IR3']
Length: 4, dtype: str

Column 'LandContour' has 4 unique categories:
<ArrowStringArray>
['Lvl', 'Bnk', 'Low', 'HLS']
Length: 4, dtype: str

Column 'Utilities' has 2 unique categories:
<ArrowStringArray>
['AllPub', 'NoSeWa']
Length: 2, dtype: str

Column 'LotConfig' has 5 unique categories:
<ArrowStringArray>
['Inside', 'FR2', 'Corner', 'CulDSac', 'FR3']
Length: 5, dtype: str

Column 'LandSlope' has 3 unique categories:
<ArrowStringArray>
['Gtl', 'Mod', 'Sev']
Length: 3, dtype: str

Column 'Neighborhood' has 25 unique categ

## Incorrect data types

In [249]:
print("---  Columns Data Types of First Dataset---")
print(df.dtypes)
print("---  Columns Data Types of Second Dataset---")
print(data.dtypes)

---  Columns Data Types of First Dataset---
Id                 int64
MSSubClass         int64
MSZoning             str
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType             str
SaleCondition        str
SalePrice          int64
Length: 81, dtype: object
---  Columns Data Types of Second Dataset---
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


## Outliers

In [250]:
print("--- Outliers in First Dataset ---")

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Standard formula limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    

    outliers_count = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()
    

    if outliers_count > 0:
        print(f"Column {col} have total {outliers_count} outliers")

print("---outliers in Second Dataset---")

num_col=data.select_dtypes(include=(['int64','float64'])).columns

for col in num_col:
    Q1=data[col].quantile(0.25)
    Q2=data[col].quantile(0.75)
    IQR=Q3-Q1

    lower_limit=Q1-1.5*IQR
    upper_limit=Q3+1.5*IQR

    outliers_count=((data[col]<lower_limit)|(data[col]>upper_limit)).sum()

    if outliers_count>0:
        print(f"Column {col} have total {outliers_count} outliers")


--- Outliers in First Dataset ---
Column MSSubClass have total 103 outliers
Column LotFrontage have total 88 outliers
Column LotArea have total 69 outliers
Column OverallQual have total 2 outliers
Column OverallCond have total 125 outliers
Column YearBuilt have total 7 outliers
Column MasVnrArea have total 96 outliers
Column BsmtFinSF1 have total 7 outliers
Column BsmtFinSF2 have total 167 outliers
Column BsmtUnfSF have total 29 outliers
Column TotalBsmtSF have total 61 outliers
Column 1stFlrSF have total 20 outliers
Column 2ndFlrSF have total 2 outliers
Column LowQualFinSF have total 26 outliers
Column GrLivArea have total 31 outliers
Column BsmtFullBath have total 1 outliers
Column BsmtHalfBath have total 82 outliers
Column BedroomAbvGr have total 35 outliers
Column KitchenAbvGr have total 68 outliers
Column TotRmsAbvGrd have total 30 outliers
Column Fireplaces have total 5 outliers
Column GarageCars have total 5 outliers
Column GarageArea have total 21 outliers
Column WoodDeckSF hav

 Noisy data


In [251]:
print("Noisy Data in First Dataset---")
#print(df.describe())
print(df.round())

print("---Noisy Data in Second Dataset---")
print(data.round())

Noisy Data in First Dataset---
        Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0        1          60       RL         65.0     8450   Pave   NaN      Reg   
1        2          20       RL         80.0     9600   Pave   NaN      Reg   
2        3          60       RL         68.0    11250   Pave   NaN      IR1   
3        4          70       RL         60.0     9550   Pave   NaN      IR1   
4        5          60       RL         84.0    14260   Pave   NaN      IR1   
...    ...         ...      ...          ...      ...    ...   ...      ...   
1455  1456          60       RL         62.0     7917   Pave   NaN      Reg   
1456  1457          20       RL         85.0    13175   Pave   NaN      Reg   
1457  1458          70       RL         66.0     9042   Pave   NaN      Reg   
1458  1459          20       RL         68.0     9717   Pave   NaN      Reg   
1459  1460          20       RL         75.0     9937   Pave   NaN      Reg   

     LandContour Uti

### Part 2 Missing-Value Treatment

## Row deletion

In [252]:
# print("--- Delete Row from First Dataset---")
# df.dropna(inplace=True)

# print("--- Delete Row from Second Dataset---")
# data.dropna(inplace=True)


## Column deletion

In [253]:
print("--- Delete Column from First Dataset---")
df.drop(columns=['Alley', 'PoolQC','MiscFeature'],inplace=True)
print(df.shape[1])

print("---Delete Column from Second Dataset---")
data.drop(columns=['deck'],inplace=True)
print(data.shape[1])

--- Delete Column from First Dataset---
78
---Delete Column from Second Dataset---
14


## Mean imputation

In [254]:
import warnings
warnings.filterwarnings('ignore')

print("----Fill missing values with mean in First Dataset----")
df['LotFrontage'].fillna(df['LotFrontage'].mean(),inplace=True)

print("----Fill missing values with mean in Second Dataset----")
data['age'].fillna(data['age'].mean(),inplace=True)

----Fill missing values with mean in First Dataset----
----Fill missing values with mean in Second Dataset----


0      22.000000
1      38.000000
2      26.000000
3      35.000000
4      35.000000
         ...    
885    39.000000
887    19.000000
888    29.869351
889    26.000000
890    32.000000
Name: age, Length: 784, dtype: float64

## Median imputation

In [255]:
print("----Fill missing values with median in First Dataset----")
df['MasVnrArea'].fillna(df['MasVnrArea'].median(),inplace=True)

print("----Fill missing values with median in Second Dataset----")

data['fare'].fillna(data['fare'].median(),inplace=True)

----Fill missing values with median in First Dataset----
----Fill missing values with median in Second Dataset----


0       7.2500
1      71.2833
2       7.9250
3      53.1000
4       8.0500
        ...   
885    29.1250
887    30.0000
888    23.4500
889    30.0000
890     7.7500
Name: fare, Length: 784, dtype: float64

## Mode imputation

In [256]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

print("----Fill Missing Values With Mode in First Dataset----")
df[['Electrical']]=imputer.fit_transform(df[['Electrical']])

print("----Fill Missing Values With Mode in Second Dataset----")
data[['embarked']] = imputer.fit_transform(data[['embarked']])



----Fill Missing Values With Mode in First Dataset----
----Fill Missing Values With Mode in Second Dataset----


##  Advanced imputation concepts 

In [257]:
from sklearn.impute import KNNImputer

imputer=KNNImputer(n_neighbors=5)
demo_cols = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']

knn_imputer = KNNImputer(n_neighbors=5)
df_knn = df.copy()
df_knn[demo_cols] = knn_imputer.fit_transform(df_knn[demo_cols])
print("✅ KNN Imputation successfully done!")

✅ KNN Imputation successfully done!


In [258]:
df.head()
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


## Part 3 Categorical Data

## • Label encoding • Ordinal encoding • One-hot encoding • 

In [259]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder

le=LabelEncoder()

print("--- Applying  Encoding on House Prices Dataset ---")

print("--- Applying Ordinal/Categorical Encoding on House Prices Dataset ---")

qual_order = [['Fa', 'TA', 'Gd', 'Ex']]

ord_enc = OrdinalEncoder(categories=qual_order)

# Column par apply karein aur save karein
df['ExterQual'] = ord_enc.fit_transform(df[['ExterQual']])
print("1. Ordinal Encoding Done (ExterQual Column):")
print(df['ExterQual'].head())


print("--- Applying OneHot Encoding on House Prices Dataset ---")


ohe = OneHotEncoder(sparse_output=False, drop='first')


zoning_encoded = ohe.fit_transform(df[['MSZoning']])

zoning_df = pd.DataFrame(zoning_encoded, columns=ohe.get_feature_names_out(['MSZoning']))

print("\n2. One-Hot Encoding Done (MSZoning Column):")
print(zoning_df.head())

--- Applying  Encoding on House Prices Dataset ---
--- Applying Ordinal/Categorical Encoding on House Prices Dataset ---
1. Ordinal Encoding Done (ExterQual Column):
0    2.0
1    1.0
2    2.0
3    1.0
4    2.0
Name: ExterQual, dtype: float64
--- Applying OneHot Encoding on House Prices Dataset ---

2. One-Hot Encoding Done (MSZoning Column):
   MSZoning_FV  MSZoning_RH  MSZoning_RL  MSZoning_RM
0          0.0          0.0          1.0          0.0
1          0.0          0.0          1.0          0.0
2          0.0          0.0          1.0          0.0
3          0.0          0.0          1.0          0.0
4          0.0          0.0          1.0          0.0


In [260]:
print("--- Applying  Encoding on Titanic Dataset ---")
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
print("--- Apply Label Encoding on Sex column ")

data["sex"]=le.fit_transform(data["sex"])
print(data['sex'].head())




class_order= [['Third','Second','First']]

ord_enc=OrdinalEncoder(categories=class_order)
data['class_Encoded'] = ord_enc.fit_transform(data[['class']])

print("Ordinal Encoding Done on Pclass Columns")

print(data[['class','class_Encoded']].head())

print("Ordinal Encoding successfully done!")


print("---OneHot Encoding---")

ohe=OneHotEncoder(sparse_output=False,drop='first')
embarked_encoded=ohe.fit_transform(data[['embarked']])

embarked_data=pd.DataFrame(embarked_encoded,columns=ohe.get_feature_names_out(['embarked']))



print(zoning_df.head())


--- Applying  Encoding on Titanic Dataset ---
--- Apply Label Encoding on Sex column 
0    1
1    0
2    0
3    0
4    1
Name: sex, dtype: int64
Ordinal Encoding Done on Pclass Columns
   class  class_Encoded
0  Third            0.0
1  First            2.0
2  Third            0.0
3  First            2.0
4  Third            0.0
Ordinal Encoding successfully done!
---OneHot Encoding---
   MSZoning_FV  MSZoning_RH  MSZoning_RL  MSZoning_RM
0          0.0          0.0          1.0          0.0
1          0.0          0.0          1.0          0.0
2          0.0          0.0          1.0          0.0
3          0.0          0.0          1.0          0.0
4          0.0          0.0          1.0          0.0


## High Cardinality

In [261]:
print(df["Neighborhood"].nunique())
freq=df["Neighborhood"].value_counts()
df["Neighborhood"]=df["Neighborhood"].map(freq)
print(df["Neighborhood"].head())

25
0    150
1     11
2    150
3     51
4     41
Name: Neighborhood, dtype: int64


In [262]:
print(data['sex'].nunique())

freq=data['sex'].value_counts()
data['sex']=data['sex'].map(freq)
print(data['sex'].head())

2
0    491
1    293
2    293
3    293
4    491
Name: sex, dtype: int64


## Rare Categories

In [263]:
counts=df["Neighborhood"].value_counts()

rare=counts[counts<20].index

df["Neighborhood"]=df["Neighborhood"].replace(rare,"Other")

print(df["Neighborhood"].head())

0      150
1    Other
2      150
3       51
4       41
Name: Neighborhood, dtype: object


## Part 4 Numerical Data

## Standardization • Min-max scaling • Robust scaling

In [264]:
from sklearn.preprocessing import StandardScaler


print("--- Applying Standardization (StandardScaler) ---")

scaler = StandardScaler()

cols_to_scale = ['LotArea', 'SalePrice']

df_scaled_array = scaler.fit_transform(df[cols_to_scale])

df_scaled = df.copy()
df_scaled[cols_to_scale] = df_scaled_array

print("\nStandardization successfully done!")
print(df_scaled[cols_to_scale].head())

print("--- Apply MinMax Scaler---")
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()


print("---Apply Robust Scaler---")
from sklearn.preprocessing import RobustScaler

scaler=RobustScaler()

--- Applying Standardization (StandardScaler) ---

Standardization successfully done!
    LotArea  SalePrice
0 -0.207142   0.347273
1 -0.091886   0.007288
2  0.073480   0.536154
3 -0.096897  -0.515281
4  0.375148   0.869843
--- Apply MinMax Scaler---
---Apply Robust Scaler---


## Log Transormation

In [265]:
data["Fare"]=np.log1p(data["fare"])

## Outlier Treatment

In [266]:
from scipy.stats import zscore

print("--- Filtering Outliers using Z-Score ---")
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df = df[(np.abs(zscore(df[num_cols], nan_policy='omit')) < 3).all(axis=1)]
print(f"Original Dataset Shape: {df.shape}")

--- Filtering Outliers using Z-Score ---
Original Dataset Shape: (796, 78)


## Part 5 Feature Engineering

## Derived Variable

In [267]:
data["FamilySize"]=data["sibsp"]+data["parch"]+1

In [268]:
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,class_Encoded,Fare,FamilySize
0,0,3,491,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,0.0,2.110213,2
1,1,1,293,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,2.0,4.280593,2
2,1,3,293,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,0.0,2.188856,1
3,1,1,293,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,2.0,3.990834,2
4,0,3,491,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0.0,2.202765,1


## Date Features

In [269]:
df["SaleAge"]=df["YrSold"]-df["YearBuilt"]


In [270]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,Fence,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,SaleAge
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,NaN,0,2,2008,WD,Normal,208500,5
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,NaN,0,9,2008,WD,Normal,223500,7
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,NaN,0,12,2008,WD,Normal,250000,8
6,7,20,RL,75.0,10084,Pave,Reg,Lvl,AllPub,Inside,...,0,0,NaN,0,8,2007,WD,Normal,307000,3
10,11,20,RL,70.0,11200,Pave,Reg,Lvl,AllPub,Inside,...,0,0,NaN,0,2,2008,WD,Normal,129500,43


## Text-length features

In [277]:
df["Street_Length"] = df["Street"].astype(str).str.len()
print("Feature Street Length Create Successfully")
print(df[["Street", "Street_Length"]].head())

Feature Street Length Create Successfully
   Street  Street_Length
0    Pave              4
2    Pave              4
4    Pave              4
6    Pave              4
10   Pave              4


## Ratio

In [278]:
data["FarePerPerson"]=data["fare"]/data["FamilySize"]

## Binning

In [280]:
data["AgeGroup"]=pd.cut(data["age"],
                      bins=[0,18,35,60,100],
                      labels=["Child","Young","Adult","Senior"])

## Interaction

In [281]:
data["AgeFare"]=data["age"]*data["fare"]

## Domain Feature

In [282]:
data["IsAlone"]=(data["FamilySize"]==1).astype(int)

## Part 6 Data Splitting

In [289]:
print("--- Split House Price Dataset---")
X_hou = df.drop('SalePrice', axis=1)
y_hou=df['SalePrice']
X_hou_train, X_hou_test, y_hou_train, y_hou_test = train_test_split(X_hou, y_hou, test_size=0.2, random_state=42)

--- Split House Price Dataset---


In [292]:
print("--- Split Titanic Dataset---")
X_tit=data.drop('survived',axis=1)
y_tit=data['survived']
X_tit_train, X_tit_test, y_tit_train, y_tit_test = train_test_split(X_tit, y_tit, test_size=0.2, random_state=42)

--- Split Titanic Dataset---


## Time-Based Split

In [294]:
df=df.sort_values("YrSold")

## Part 7 Data Leakage

## Part 8 Scikit Learn

In [295]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
numeric_pipeline=Pipeline([

("imputer",SimpleImputer(strategy="median")),

("scaler",StandardScaler())

])

categorical_pipeline=Pipeline([

("imputer",SimpleImputer(strategy="most_frequent")),

("encoder",OneHotEncoder(handle_unknown="ignore"))

])

preprocessor=ColumnTransformer([

("num",numeric_pipeline,num_cols),

("cat",categorical_pipeline,cat_cols)

])